# 18a HKO certified upper-tail market price panel

This notebook builds the market-implied probability panel for the formally certified Hong Kong highest-temperature upper-tail contracts produced by `17l`.

Input:

`data/processed/17l_hko_highest_upper_tail_formally_certified_subset_deduped.csv`

Outputs:

- `data/processed/18a_hko_certified_upper_tail_market_price_panel.csv`
- `data/processed/18a_hko_certified_upper_tail_price_coverage_summary.csv`
- `data/raw/polymarket_clob_price_history_18a/*.json`
- `docs/research_outputs/18a_hko_certified_upper_tail_price_panel_report.md`

Interpretation: these price-history observations are used as market-implied probability observations. They are not yet a fully executable trading backtest because execution requires a separate bid/ask and liquidity audit.

In [ ]:
# Run the paired script from the repository root.
from pathlib import Path
import subprocess

repo = Path.cwd().resolve()
while not (repo / ".git").exists() and repo.parent != repo:
    repo = repo.parent

script = repo / "scripts" / "18a_hko_certified_upper_tail_price_panel.py"
print("Repository root:", repo)
print("Script:", script)
assert script.exists(), f"Missing script: {script}"

result = subprocess.run(["python3", str(script)], cwd=str(repo), text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Inspect outputs

After the retrieval script completes, inspect the coverage summary and the first rows of the price panel.

In [ ]:
import pandas as pd
from pathlib import Path

repo = Path.cwd().resolve()
while not (repo / ".git").exists() and repo.parent != repo:
    repo = repo.parent

coverage_path = repo / "data" / "processed" / "18a_hko_certified_upper_tail_price_coverage_summary.csv"
panel_path = repo / "data" / "processed" / "18a_hko_certified_upper_tail_market_price_panel.csv"

coverage = pd.read_csv(coverage_path)
print("Coverage shape:", coverage.shape)
print(coverage["price_history_status"].value_counts(dropna=False))
display(coverage)

panel = pd.read_csv(panel_path)
print("Panel shape:", panel.shape)
if not panel.empty:
    display(panel.head(20))
    display(panel.groupby(["event_date", "threshold_K", "certified_contract_id"]).size().reset_index(name="n_price_rows"))
else:
    print("No price rows retrieved. Check yes-token extraction and raw JSON evidence.")

## Next step

If the panel is non-empty, the next notebook should join these market-implied probabilities to realised HKO outcomes. If the panel is empty for some contracts, use the coverage summary to diagnose whether the issue is token extraction or CLOB price-history availability.